In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv("breast_cancer.csv")

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 455
Testing rows: 114


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

numeric_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ])
}

print("Models ready:", list(models.keys()))

Models ready: ['Logistic Regression', 'Random Forest']


In [6]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, recall_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Target 0 represents malignant cases in this dataset
malignant_recall = make_scorer(
    recall_score,
    pos_label=0
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "malignant_recall": malignant_recall
}

cv_results = []

for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    cv_results.append({
        "Model": name,
        "CV Accuracy Mean": scores["test_accuracy"].mean(),
        "CV Accuracy Std": scores["test_accuracy"].std(),
        "CV Balanced Accuracy Mean": scores["test_balanced_accuracy"].mean(),
        "CV Malignant Recall Mean": scores["test_malignant_recall"].mean()
    })

cv_table = pd.DataFrame(cv_results)

print(cv_table.round(4).to_string(index=False))

              Model  CV Accuracy Mean  CV Accuracy Std  CV Balanced Accuracy Mean  CV Malignant Recall Mean
Logistic Regression            0.9780           0.0098                     0.9753                    0.9647
      Random Forest            0.9626           0.0179                     0.9619                    0.9588


In [7]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)

test_results = []
fitted_models = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    fitted_models[name] = pipeline

    test_results.append({
        "Model": name,
        "Test Accuracy": accuracy_score(y_test, predictions),
        "Test Balanced Accuracy": balanced_accuracy_score(y_test, predictions),
        "Malignant Recall": recall_score(
            y_test, predictions, pos_label=0
        ),
        "Malignant Precision": precision_score(
            y_test, predictions, pos_label=0, zero_division=0
        )
    })

test_table = pd.DataFrame(test_results)

print(test_table.round(4).to_string(index=False))

              Model  Test Accuracy  Test Balanced Accuracy  Malignant Recall  Malignant Precision
Logistic Regression         0.9825                  0.9812            0.9762               0.9762
      Random Forest         0.9561                  0.9504            0.9286               0.9512


In [8]:
for name, pipeline in fitted_models.items():
    predictions = pipeline.predict(X_test)

    print("\n", name)
    print("Rows = actual class; columns = predicted class")
    print("Class order: 0 = malignant, 1 = benign")
    print(confusion_matrix(y_test, predictions, labels=[0, 1]))


 Logistic Regression
Rows = actual class; columns = predicted class
Class order: 0 = malignant, 1 = benign
[[41  1]
 [ 1 71]]

 Random Forest
Rows = actual class; columns = predicted class
Class order: 0 = malignant, 1 = benign
[[39  3]
 [ 2 70]]


In [9]:
model_to_test = fitted_models["Logistic Regression"]
invalid_input = X_test.iloc[[0]].drop(columns=[X_test.columns[0]])

try:
    model_to_test.predict(invalid_input)
    raise AssertionError("Expected missing-feature input to fail.")
except (ValueError, KeyError) as error:
    print("Negative test passed: missing feature was rejected.")
    print("Error type:", type(error).__name__)

Negative test passed: missing feature was rejected.
Error type: ValueError


In [10]:
from pathlib import Path

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

cv_table.to_csv(
    reports_dir / "day3_cross_validation_results.csv",
    index=False
)

test_table.to_csv(
    reports_dir / "day3_test_results.csv",
    index=False
)

print("Results saved successfully.")

Results saved successfully.
